# Trabajo Final Integrador — Estimación de la producción mensual de petróleo en pozos shale de Vaca Muerta

**Versión para VS Code** del notebook de Colab (`00_colab_original.ipynb`), adaptada a las correcciones de la devolución de la propuesta.

| Corrección de la devolución | Cómo se resuelve en este notebook |
|---|---|
| Reproducibilidad del R² | Snapshot versionado con sha256 (`data/snapshots/`) y se reproduce la configuración original (sección 3.1) |
| Variabilidad por semilla | Validación repetida: 10 separaciones por pozo → promedio ± desvío (3.2) |
| Validación temporal | Entrenamiento con los meses anteriores y prueba con los últimos meses (3.4) |
| "Mes" no aporta | Se reemplaza por la **edad del pozo** (meses desde la primera producción) |
| TEF solo se conoce al cerrar el mes | Tres versiones: **v1** (base), **v2** (descriptiva, con TEF) y **v2_notef** (pronóstico) |
| Comparación justa entre versiones | **Holdout fijo por pozo** (hash del idpozo): el mismo para todas las versiones (3.3) |
| Declinación mes a mes | Curva real vs. estimada de pozos del holdout (sección 4) |

> Este notebook es el **análisis exploratorio**. Las versiones oficiales del modelo (con `metadata.json` y promoción) las genera el servicio `training/` con el mismo código (`vm_ml`).

**Cómo correrlo:** `.\tasks.ps1 setup` y elegir el kernel **TF Vaca Muerta (.venv)** en VS Code. Antes, generar el snapshot con `.\tasks.ps1 data-arcgis` y `.\tasks.ps1 data-build`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit, train_test_split

from vm_ml import config
from vm_ml.data import load_snapshot, population, month_label
from vm_ml.features import model_frame, make_pipeline, feature_columns
from vm_ml.evaluation import metrics, holdout_mask, repeated_group_validation, temporal_validation

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

snapshots = sorted((ROOT / "data" / "snapshots").glob("*.csv.gz"))
if snapshots:
    SNAPSHOT = snapshots[-1]
else:
    SNAPSHOT = sorted((ROOT / "data" / "synthetic").glob("*.csv.gz"))[-1]
    print("ATENCIÓN: usando datos SINTÉTICOS. No reportar estas métricas.")
print("Snapshot:", SNAPSHOT.name)

## 1. Datos y reproducibilidad

Fuente: servicio ArcGIS REST de la Secretaría de Energía (y, si está disponible, la serie histórica y el padrón de pozos de datos.gob.ar). El script `scripts/data_vm.py` descarga Vaca Muerta / no convencional / shale y arma un **snapshot** con su manifest (sha256, fuentes, rango de meses). `load_snapshot` verifica el hash: si el archivo cambió, falla.

In [ ]:
df, manifest = load_snapshot(SNAPSHOT)
print(json.dumps(manifest, indent=2, ensure_ascii=False))

In [ ]:
print("Registros shale:", f"{len(df):,}", "| pozos:", f"{df['idpozo'].nunique():,}")
display(df["tipopozo"].value_counts(dropna=False).to_frame("registros"))

pop = population(df)
resumen = pd.DataFrame({
    "Etapa": ["Vaca Muerta - No convencional - SHALE", "Petrolíferos", "Petrolíferos en extracción efectiva"],
    "Registros": [len(df), int((df["tipopozo"] == "Petrolífero").sum()), len(pop)],
})
display(resumen)
print("Pozos en la población:", f"{pop['idpozo'].nunique():,}")
print("Meses:", month_label(int(pop['mes_idx'].min())), "a", month_label(int(pop['mes_idx'].max())),
      f"({pop['mes_idx'].nunique()} meses)")
print(f"prod_pet = 0: {(pop['prod_pet'] == 0).sum():,} ({(pop['prod_pet'] == 0).mean():.2%})")
print(f"Edad del pozo: fuente padrón {(pop['fuente_edad'] == 'padron').mean():.1%}, "
      f"censurada {pop['edad_censurada'].mean():.1%}")

## 2. Exploración

Además de las variables del notebook original se analiza la **edad del pozo** (`edad_meses`). Si la edad está *censurada* (el pozo ya producía en el primer mes disponible y no hay padrón), su valor es un mínimo: por eso conviene ampliar la ventana de meses o usar el padrón.

In [ ]:
candidatas = ["mes", "tef", "profundidad", "edad_meses", "tipoextraccion", "empresa", "areayacimiento", "prod_pet"]
calidad = pd.DataFrame({
    "faltantes": pop[candidatas].isna().sum(),
    "porcentaje_faltantes": (pop[candidatas].isna().mean() * 100).round(2),
    "valores_unicos": pop[candidatas].nunique(),
})
display(calidad)
display(pop["prod_pet"].describe(percentiles=[.25, .5, .75, .9, .95, .99]).to_frame())
display(pop["tipoextraccion"].value_counts().to_frame("registros"))

In [ ]:
num = ["mes", "tef", "profundidad", "edad_meses", "prod_pet"]
print("Pearson (asociación lineal):")
display(pop[num].corr(method="pearson").round(3))
print("Spearman (asociación monótona, más robusta a la asimetría de prod_pet):")
display(pop[num].corr(method="spearman").round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
p99 = pop["prod_pet"].quantile(0.99)
axes[0].hist(pop.loc[pop["prod_pet"] <= p99, "prod_pet"], bins=50)
axes[0].set_title("Producción mensual (hasta P99)")
axes[0].set_xlabel("m³/mes"); axes[0].set_ylabel("registros")

por_edad = pop[~pop["edad_censurada"]].groupby("edad_meses")["prod_pet"].agg(["median", "count"])
por_edad = por_edad[por_edad["count"] >= 20]
axes[1].plot(por_edad.index, por_edad["median"], marker="o")
axes[1].set_title("Mediana de producción según la edad del pozo\n(solo edades no censuradas, ≥20 registros)")
axes[1].set_xlabel("meses desde la primera producción"); axes[1].set_ylabel("m³/mes")
plt.tight_layout(); plt.show()

## 3. Metodología y resultados

Tres versiones del árbol de decisión, con regresión lineal y "predecir el promedio" como referencias:

| Versión | Variables | Uso |
|---|---|---|
| v1 | mes, TEF, profundidad, sistema de extracción | Línea base (propuesta) |
| v2 | TEF, profundidad, **edad**, sistema de extracción, **operadora**, **área** | Descriptiva (requiere el TEF del mes) |
| v2_notef | profundidad, **edad**, sistema de extracción, operadora, área | **Pronóstico** (sin información del mes cerrado) |

### 3.1 Reproducción de la configuración original (respuesta al punto 1 de la devolución)

Notebook original: `DecisionTreeRegressor(max_depth=6, min_samples_leaf=20)` y `GroupShuffleSplit(test_size=0.2, random_state=42)` por `idpozo`. Se compara con una separación **aleatoria por fila**, que mezcla meses del mismo pozo en entrenamiento y prueba y por eso infla el R².

In [ ]:
X1, y1, g1, m1 = model_frame(df, "v1")
original = {"max_depth": 6, "min_samples_leaf": 20}
filas = []

tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(X1, y1, g1))
mdl = make_pipeline("v1", "tree", original).fit(X1.iloc[tr], y1.iloc[tr])
filas.append({"separación": "por pozo, semilla 42 (notebook original)", **metrics(y1.iloc[te], mdl.predict(X1.iloc[te]))})

mdl = make_pipeline("v1", "tree", {}).fit(X1.iloc[tr], y1.iloc[tr])
filas.append({"separación": "por pozo, semilla 42, árbol SIN límites", **metrics(y1.iloc[te], mdl.predict(X1.iloc[te]))})

Xtr, Xte, ytr, yte = train_test_split(X1, y1, test_size=0.2, random_state=42)
mdl = make_pipeline("v1", "tree", original).fit(Xtr, ytr)
filas.append({"separación": "aleatoria por fila (filtra información)", **metrics(yte, mdl.predict(Xte))})
display(pd.DataFrame(filas))

### 3.2 Validación repetida (10 separaciones por pozo, sobre el conjunto de desarrollo)

Con un semestre de datos, una sola separación es ruidosa: se informan promedio y desvío.

In [ ]:
is_hold_by_model = {}
tabla, por_semilla = [], {}
for name in config.FEATURE_SETS:
    X, y, g, mi = model_frame(df, name)
    hold = holdout_mask(g)
    is_hold_by_model[name] = hold
    Xd, yd, gd = X[~hold], y[~hold], g[~hold]
    params = config.DEFAULT_TREE_PARAMS[name]
    for kind in ["dummy_mean", "linear", "tree"]:
        r = repeated_group_validation(lambda k=kind: make_pipeline(name, k, params if k == "tree" else None), Xd, yd, gd)
        s = r["summary"]
        tabla.append({"versión": name, "modelo": kind,
                      "MAE": f"{s['mae']['mean']:,.1f} ± {s['mae']['std']:,.1f}",
                      "RMSE": f"{s['rmse']['mean']:,.1f} ± {s['rmse']['std']:,.1f}",
                      "R²": f"{s['r2']['mean']:.3f} ± {s['r2']['std']:.3f}"})
        if kind == "tree":
            por_semilla[name] = [run["mae"] for run in r["runs"]]
display(pd.DataFrame(tabla))

plt.figure(figsize=(8, 4))
for name, maes in por_semilla.items():
    plt.plot(config.CV_SEEDS, maes, marker="o", label=name)
plt.xlabel("semilla de la separación"); plt.ylabel("MAE [m³/mes]")
plt.title("Sensibilidad del MAE a la semilla (árbol)"); plt.legend(); plt.show()

### 3.3 Holdout fijo por pozo y control de sobreajuste

El holdout se define por `sha256(salt:idpozo) % 100 < 20`: no depende de una semilla, y es el mismo para todas las versiones y entrenamientos. El modelo final se entrena solo con el conjunto de desarrollo.

In [ ]:
filas, modelos_finales = [], {}
for name in config.FEATURE_SETS:
    X, y, g, _ = model_frame(df, name)
    hold = is_hold_by_model[name]
    mdl = make_pipeline(name, "tree", config.DEFAULT_TREE_PARAMS[name]).fit(X[~hold], y[~hold])
    modelos_finales[name] = mdl
    for conjunto, mask in [("train", ~hold), ("holdout", hold)]:
        filas.append({"versión": name, "conjunto": conjunto, "pozos": g[mask].nunique(),
                      **metrics(y[mask], mdl.predict(X[mask]))})
display(pd.DataFrame(filas))

### 3.4 Validación temporal

Se entrena con los meses anteriores al corte y se prueba con los últimos meses: en pozos ya conocidos (*mismos_pozos*) y en pozos del holdout (*pozos_nuevos*, lo más exigente).

In [ ]:
filas = []
for name in config.FEATURE_SETS:
    X, y, g, mi = model_frame(df, name)
    r = temporal_validation(lambda n=name: make_pipeline(n, "tree", config.DEFAULT_TREE_PARAMS[n]),
                            X, y, g, mi, is_hold_by_model[name])
    for caso in ["mismos_pozos", "pozos_nuevos"]:
        if caso in r:
            filas.append({"versión": name, "caso": caso, "corte": month_label(r["cutoff_mes_idx"]), **r[caso]})
display(pd.DataFrame(filas))

## 4. Declinación mes a mes

La devolución mostró que el árbol pone los meses de un pozo en la misma rama y la estimación queda casi plana mientras la producción real cae. Se grafican pozos del holdout con al menos 5 meses de datos.

In [ ]:
X2, y2, g2, m2 = model_frame(df, "v2")
pozos = (g2[is_hold_by_model["v2"]].value_counts())
pozos = pozos[pozos >= 5].index[:3]

fig, axes = plt.subplots(1, len(pozos), figsize=(5 * len(pozos), 4), squeeze=False)
for ax, pozo in zip(axes[0], pozos):
    real = None
    for name, mdl in modelos_finales.items():
        X, y, g, mi = model_frame(df, name)
        mask = (g == pozo).to_numpy()
        orden = np.argsort(mi[mask].to_numpy())
        meses = [month_label(v) for v in mi[mask].to_numpy()[orden]]
        if real is None:
            real = y[mask].to_numpy()[orden]
            ax.plot(meses, real, color="black", marker="o", label="real")
        ax.plot(meses, mdl.predict(X[mask])[orden], linestyle="--", marker=".", label=name)
    info = pop[pop["idpozo"] == pozo].iloc[0]
    ax.set_title(f"Pozo {pozo} · {info['empresa']}\n{info['areayacimiento']}", fontsize=9)
    ax.tick_params(axis="x", rotation=45); ax.set_ylabel("m³/mes")
axes[0][0].legend(); plt.tight_layout(); plt.show()

In [ ]:
mdl = modelos_finales["v2"]
nombres = mdl.named_steps["preprocesamiento"].get_feature_names_out()
imp = pd.DataFrame({"variable": nombres, "importancia": mdl.named_steps["modelo"].feature_importances_})
imp = imp.sort_values("importancia", ascending=False).head(15)
display(imp)
imp.sort_values("importancia").plot.barh(x="variable", y="importancia", figsize=(9, 6), legend=False,
                                         title="Variables más usadas por el árbol (v2)")
plt.show()

## 5. Interpretación (completar con los resultados reales)

> Esta sección se redacta con las tablas de arriba **ejecutadas sobre el snapshot real**, no sobre los datos sintéticos.

- **Reproducción del R² original (3.1):** …
- **Variabilidad por semilla (3.2):** …
- **Efecto de la edad, la operadora y el área (v1 → v2):** …
- **Costo de no usar el TEF (v2 → v2_notef):** …
- **Validación temporal (3.4):** …
- **Declinación (4):** …
- **Versión elegida para el servicio y por qué:** …